# Phase 1: CLIP Zero-Shot Unlearning (NegMerge)
In this notebook, we evaluate the NegMerge algorithm on the Vision-Language Zero-Shot unlearning scenario.
We use the pre-finetuned models downloaded in the `NegMerge_Checkpoints` directory to avoid costly fine-tuning.
We will compare the unlearning performance of merging 10 models versus merging 30 models.

In [1]:
import warnings
warnings.filterwarnings('ignore')
import os
import sys
import torch
import json
import copy

# Add the CLIP_MU path so we can import NegMerge's evaluation tools
sys.path.append(os.path.abspath('./CLIP_MU'))
from src.args import parse_arguments
from src.task_vectors import NonLinearTaskVector
from src.eval import evaluate_task_vector, evaluate_task_vector_at_coef
from src.utils import find_optimal_coef


## 1. Environment and Checkpoint Discovery

In [2]:
CHECKPOINT_DIR = 'NegMerge_Checkpoints'
PRETRAINED_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 'zeroshot.pt')
ACCURACY_FILE = os.path.join(CHECKPOINT_DIR, 'zeroshot_accuracies.json')

finetuned_models = [f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('_finetuned.pt')]
print(f"Found {len(finetuned_models)} finetuned models.")

finetuned_paths = [os.path.join(CHECKPOINT_DIR, f) for f in finetuned_models]

with open(ACCURACY_FILE) as f:
    pretrained_accuracies = json.load(f)
print("Pre-trained Original Accuracies:", pretrained_accuracies)


Found 30 finetuned models.
Pre-trained Original Accuracies: {'Cars': 0.5964432284541724, 'CarsVal': 0.5958230958230958, 'DTD': 0.4441489361702128, 'DTDVal': 0.43882978723404253, 'EuroSAT': 0.45185185185185184, 'EuroSATVal': 0.4507407407407407, 'GTSRB': 0.32557403008709423, 'GTSRBVal': 0.32995495495495497, 'MNIST': 0.4825, 'MNISTVal': 0.4818, 'RESISC45': 0.6022222222222222, 'RESISC45Val': 0.6158730158730159, 'SUN397': 0.6318891687657431, 'SUN397Val': 0.6327455919395466, 'SVHN': 0.3160725261216964, 'SVHNVal': 0.292, 'ImageNetVal': 0.6666, 'ImageNet': 0.63338}


## 2. NegMerge Implementation

In [3]:
def apply_negmerge(checkpoint_paths, pretrained_checkpoint):
    print(f"Applying NegMerge over {len(checkpoint_paths)} models...")
    for idx, cp in enumerate(checkpoint_paths):
        tv = NonLinearTaskVector(pretrained_checkpoint, cp)
        if idx == 0:
            task_vector = tv
            merged_vector = {k: torch.zeros_like(v) for k, v in tv.vector.items()}
            mask = {k: torch.zeros_like(v) for k, v in tv.vector.items()}
        
        for key in task_vector.vector.keys():
            merged_vector[key] += tv.vector[key]
            mask[key] += torch.sign(tv.vector[key])

    # Now apply the consensus mask
    sample_state_dict = torch.load(checkpoint_paths[0], map_location='cpu')
    for key in sample_state_dict.keys():
        consistency_mask = torch.abs(mask[key]) == len(checkpoint_paths)
        task_vector.vector[key] = torch.where(consistency_mask, merged_vector[key] / len(checkpoint_paths),
                                              torch.zeros_like(merged_vector[key]))
    return task_vector


## 3. Evaluation: 10 vs 30 Models

In [4]:
dataset = "Cars"
control_dataset = "MNIST"

def evaluate_negmerge(merged_task_vector, args):
    val_metrics = evaluate_task_vector(
        -merged_task_vector,
        PRETRAINED_CHECKPOINT,
        args,
    )

    optimal_coef = find_optimal_coef(
        val_metrics,
        metric=f"{dataset}Val:top1",
        minimize=True,
        control_metric=f"{control_dataset}Val:top1",
        control_metric_threshold=0.95 * pretrained_accuracies[control_dataset + "Val"],
    )

    args.eval_datasets = [dataset]
    args.control_dataset = control_dataset
    
    test_metrics = evaluate_task_vector_at_coef(
        -merged_task_vector,
        PRETRAINED_CHECKPOINT,
        args,
        optimal_coef,
    )
    
    return test_metrics, optimal_coef


In [5]:
# Initialize arguments
sys.argv=["dummy"]
args = parse_arguments()
args.model = "ViT-B-32" # Change if different backbone
args.eval_datasets = [dataset + "Val"]
args.control_dataset = control_dataset + "Val"

# 1. Evaluate 10 models
print("=== Evaluating 10 Models ===")
tv_10 = apply_negmerge(finetuned_paths[:10], PRETRAINED_CHECKPOINT)
metrics_10, coef_10 = evaluate_negmerge(tv_10, copy.deepcopy(args))
print(f"10 Models -> Accuracy on Cars: {metrics_10[f'{dataset}:top1']:.4f}, Accuracy on ImageNet: {metrics_10[f'{control_dataset}:top1']:.4f}, Coef: {coef_10}")

# 2. Evaluate 30 models
print("\n=== Evaluating 30 Models ===")
tv_30 = apply_negmerge(finetuned_paths[:30], PRETRAINED_CHECKPOINT)
metrics_30, coef_30 = evaluate_negmerge(tv_30, copy.deepcopy(args))
print(f"30 Models -> Accuracy on Cars: {metrics_30[f'{dataset}:top1']:.4f}, Accuracy on ImageNet: {metrics_30[f'{control_dataset}:top1']:.4f}, Coef: {coef_30}")


=== Evaluating 10 Models ===
Applying NegMerge over 10 models...
Evaluating for scaling coefficient 0.00
Evaluating on CarsVal
Classification head for ViT-B-32 on CarsVal exists at NegMerge_Checkpoints/head_CarsVal.pt
Loading classification head from NegMerge_Checkpoints/head_CarsVal.pt
Note: Loading Stanford Cars from reliable HuggingFace mirror (uses cache after first download)...


100%|██████████| 7/7 [00:46<00:00,  6.58s/it]


Done evaluating on CarsVal. Accuracy: 61.67%
CarsVal Top-1 accuracy: 0.6167
Evaluating on MNISTVal
Did not find classification head for ViT-B-32 on MNISTVal at None/None/ViT-B-32/head_MNISTVal.pt, building one from scratch.
Loading ViT-B-32 pre-trained weights.
Building classification head.


100%|██████████| 40/40 [00:23<00:00,  1.73it/s]


Done evaluating on MNISTVal. Accuracy: 45.76%
MNISTVal Top-1 accuracy: 0.4576
Evaluating for scaling coefficient 0.05
Evaluating on CarsVal
Classification head for ViT-B-32 on CarsVal exists at NegMerge_Checkpoints/head_CarsVal.pt
Loading classification head from NegMerge_Checkpoints/head_CarsVal.pt
Note: Loading Stanford Cars from reliable HuggingFace mirror (uses cache after first download)...


...

Done evaluating on CarsVal. Accuracy: 42.87%
CarsVal Top-1 accuracy: 0.4287
Evaluating on MNISTVal
Did not find classification head for ViT-B-32 on MNISTVal at None/None/ViT-B-32/head_MNISTVal.pt, building one from scratch.
Loading ViT-B-32 pre-trained weights.
Building classification head.


100%|██████████| 40/40 [00:23<00:00,  1.71it/s]


Done evaluating on MNISTVal. Accuracy: 43.82%
MNISTVal Top-1 accuracy: 0.4382
Evaluating for scaling coefficient 0.45
Evaluating on CarsVal
Classification head for ViT-B-32 on CarsVal exists at NegMerge_Checkpoints/head_CarsVal.pt
Loading classification head from NegMerge_Checkpoints/head_CarsVal.pt
Note: Loading Stanford Cars from reliable HuggingFace mirror (uses cache after first download)...


 14%|█▍        | 1/7 [00:09<00:56,  9.42s/it]


RuntimeError: Caught RuntimeError in DataLoader worker process 1.
Original Traceback (most recent call last):
  File "/opt/homebrew/Caskroom/miniconda/base/envs/negmerge-env/lib/python3.10/site-packages/torch/utils/data/_utils/worker.py", line 351, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/opt/homebrew/Caskroom/miniconda/base/envs/negmerge-env/lib/python3.10/site-packages/torch/utils/data/_utils/fetch.py", line 55, in fetch
    return self.collate_fn(data)
  File "/opt/homebrew/Caskroom/miniconda/base/envs/negmerge-env/lib/python3.10/site-packages/torch/utils/data/_utils/collate.py", line 398, in default_collate
    return collate(batch, collate_fn_map=default_collate_fn_map)
  File "/opt/homebrew/Caskroom/miniconda/base/envs/negmerge-env/lib/python3.10/site-packages/torch/utils/data/_utils/collate.py", line 211, in collate
    return [
  File "/opt/homebrew/Caskroom/miniconda/base/envs/negmerge-env/lib/python3.10/site-packages/torch/u